In [27]:
import os
import re
import glob
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Pandas display settings
pd.set_option("display.max_rows", 120)   # hiển thị tối đa 120 dòng
pd.set_option("display.width", 160)      # tăng độ rộng khi in DataFrame


# Data path

DATA_RAW = Path("D:\\nckh_qaaa\\data\\raw")


Mỗi thuật ngữ được giải thích theo 3 lớp:
> **(1) Nghĩa thông thường → (2) Nghĩa trong dataset cụ thể này → (3) Ví dụ số liệu thật.**

---

## A. Thuật ngữ về THIẾT BỊ & ĐO LƯỜNG

### 1. iButton (DS1922L / DS1923)
- **Nghĩa chung**: một "nút" ghi dữ liệu (data logger) hình đồng xu nhỏ của hãng Dallas/Maxim, tự ghi nhiệt độ vào bộ nhớ trong theo chu kỳ, không cần dây.
- **Trong dataset**: đây chính là các **sensor** được thả *vào trong thùng* cùng xoài. Có 62 cái cho 31 thùng.
  - `DS1922L` = bản chỉ đo **nhiệt độ** → file tên kết thúc bằng `_Temp`.
  - `DS1923` = bản đo **nhiệt độ + độ ẩm** → file tên kết thúc bằng `_Temp_Hum`.
- **Điểm mấu chốt khi explore**: mỗi file log = **một iButton tại một vị trí**, không phải một "cảm biến" trừu tượng.

### 2. Data logger / Sensor_ID
- **Sensor_ID**: mã định danh duy nhất của từng thiết bị đo (thường nằm ở dòng header của file iButton gốc). Khi merge nhiều chặng (`00_All_Recording...`), Sensor_ID cho biết các file Step rời rạc có phải **cùng một con cảm biến** hay không.

### 3. Sampling step (bước nhảy thời gian / chu kỳ lấy mẫu)
- Chuỗi thời gian cách đều nhau bao lâu thì ghi 1 điểm. Dataset này = **5 phút** (README xác nhận, notebook kiểm chứng lại trên cả 342 file).
- Nghĩa là 1 giờ có 12 dòng dữ liệu; 1 ngày = 288 dòng. Dùng để tính kỳ vọng số dòng và phát hiện gap.

### 4. Resolution (độ phân giải) vs Uncertainty (độ bất định ±)
- **Resolution 0.5°C**: giá trị nhỏ nhất thiết bị *phân biệt được* → số liệu nhiệt độ sẽ **nhảy bậc 0.5** (31.5, 32.0... chứ không bao giờ ra 31.7). Đừng nhầm bước nhảy 0.5°C này với "gap" hay "bất thường".
- **Uncertainty ±0.6°C (k=2)**: khoảng tin cậy ~95% quanh giá trị đọc. Nghĩa vật lý: nếu log ghi `31.6`, nhiệt độ *thực* nằm đâu đó trong `[31.0 ; 32.2]`.
  - Hệ quả khi explore: chênh lệch giữa 2 cảm biến < 0.6–1.2°C **chưa chắc là khác biệt thật** — phải so với độ bất định trước khi kết luận.
- **k = 2**: hệ số mở rộng (coverage factor) trong đo lường học, k=2 ≈ mức tin cậy 95%.

### 5. Tiếp xúc (contact) vs Không khí (air) vs Khoang ngoài
- `T_mangoes`: iButton **áp/bỏ trong quả** → đo khối thịt quả, phản ánh nhiệt tâm quả, trễ hơn không khí.
- `T_Air`: iButton đo **không khí lồng trong thùng carton**.
- Khoang ngoài: nhiệt độ **bên ngoài thùng** (kho lạnh INRAE 16°C/21°C, sàn bay...) — môi trường lớn.

---

## B. Thuật ngữ về CHUỖI LẠNH (COLD CHAIN)

### 6. Cold chain (chuỗi lạnh)
Toàn bộ hành trình giữ lạnh từ vườn → người ăn. Mỗi "khâu" là một **Step** (0→10): Packing House → Tiền lạnh → Xe → Sân bay đi → Bay → Sân bay đến → Lưu kho...
Mục tiêu đề tài: tìm khâu nào để nhiệt độ "trượt" khỏi dải an toàn (>~8–10°C gây hỏng/qu chín nhanh).

### 7. ULD (Unit Load Device)
- Container/kiện chuẩn dùng chất hàng lên máy bay thân rộng. README: **1 ULD chứa 148 thùng carton**, trong đó **31 thùng được gắn cảm biến** đặt ở các vị trí khác nhau.
- Vì vậy Box_ID `X_Y_Z` mô tả **vị trí hình học trong ULD** (xem mục C).

### 8. Pre-cooling (tiền lạnh, Step 1 – Cold Storage)
Làm lạnh nhanh ngay sau thu hoạch để "rút" nhiệt đồng ruộng (field heat) khỏi quả trước khi xếp lên máy bay. Step dài nhất (~20h) trong timeline.

### 9. Chilling injury (CI – tổn thương lạnh)
Hỏng do bảo quản quá lạnh (<~5–8°C với cây nhiệt đới như xoài): rám vỏ, chín ép, vị lạ. Là lý do thí nghiệm chọn 2 mức lưu kho **16°C vs 21°C** (an toàn, gần ngưỡng CI) để đối chiếu tốc độ suy giảm phẩm chất.

---

## C. Thuật ngữ về CẤU TRÚC THỰC THỂ

### 10. Box_ID dạng `X_Y_Z` (ví dụ `1_1_1`, `3_3_12`)
Không phải "tầng/vị trí" tùy tiện — **đây là tọa độ pallet trong ULD** (được cộng sự Pháp xác nhận trong README/script):
```
X = hàng dọc (row)      : 1 … 4   (1 = sát cửa/lối đi ←→ 4 = sâu trong ULD)
Y = hàng ngang (column) : 1 … 4   (trái ↔ phải)
Z = cột đứng (stack)    : 1 … 12  (1 = đáy ←→ 12 = đỉnh)
```
- 4×4×12 = 192 ô khả dĩ, thực tế 148 thùng, 31 thùng có cảm biến.
- **Ý nghĩa khi explore**: X,Y,Z quyết định thùng chịu gió lạnh/gió nóng thế nào → vì sao README nói có "10 box đỉnh, 9 box giữa chiều cao, 5 box đáy..." (chọn theo vị trí cực đoan để bắt gradient nhiệt).
- ⚠️ Đừng nhầm `X_Y_Z` trong file sensor (vị trí ULD) với số `1_/2_/3_/4_` đầu file quality (**configuration** — xem mục E).

### 11. Mango_ID dạng `BOX_NAME` ví dụ `1_1_1/1-1A`
Chuỗi ghép trong file màu/khối lượng, tách được thành 3 phần:
```
1_1_1   /   1     -   1A
 │        │         └─ vị trí đo trên quả (1..6) + mã lô ảnh
 │        └─ quả thứ # trong thùng (1..14)
 └─ Box_ID (thùng)
```
File weight dùng `M_1_1_1/7` = khối lượng **quả số 7 trong thùng (1,1,1)**. Mỗi thùng 14 quả, chia 2 nhóm 7 bởi một vách ngăn (partition).

### 12. Entity hierarchy (phân cấp thực thể)
Sơ đồ "trong-này-chứa-trong-kia": Shipment → Pallet/ULD → Box → Mango → Sensor. Mọi bảng khi explore nên giữ đủ khóa ngoại này (dạng `box`, `mango`) để group-by/pivot được mà không gộp nhầm.

---

## D. Thuật ngữ về PHẨM CHẤT XOÀI (QUALITY ATTRIBUTES)

### 13. Khối lượng M (Mass) — đơn vị gam
Xoài mất nước → hao hụt khối lượng (weight loss). So sánh M_D15/M_D0 (% thất thoát) là chỉ số thương mại quan trọng. Cân KERN ±0.008g — rất nhỏ so với quả ~350g, tức nhiễu cân bỏ qua được.

### 14. Màu CIE L\*a\*b\* (spectrophotometer CM-700d)
Không gian màu chuẩn quốc tế, 3 trục độc lập:
| Trục | Nghĩa | Xoài chín thì... |
|---|---|---|
| **L\*** | Độ sáng, 0 (đen) → 100 (trắng) | thay đổi nhẹ |
| **a\*** | **−a = xanh lá**, **+a = đỏ/đỏ cam** | tăng (lá → sắc đỏ/cam) |
| **b\*** | **−b = xanh dương**, **+b = vàng** | **tăng mạnh** (xoài chuyển vàng) |
- Ví dụ thật (file color): `1_1_1/1-1A`: b\* = 34.09 (D0) → 51.55 (D9) → 46.85 (D15): quả vàng dần rồi hơi chững — tín hiệu chín.
- Đo **6 vị trí/quả** (các đuôi tệp  `-1A ... -6F`), trung bình 3 lần đo liên tiếp.
- Tổng hợp thành chỉ số vàng hóa thường dùng: `Chromaticity = √(a*² + b*²)` hoặc `hue angle = atan2(b*, a*)`.

### 15. pH
Độ acid (thang log, 0–14 càng thấp càng chua). Xoài chín: acid hữu cơ giảm → **pH tăng nhẹ** (≈4.0 → 4.5+). Máy Starter 300 ±0.03.

### 16. TSS / °Brix (Total Soluble Solids)
Tổng chất rắn hòa tan (chủ yếu đường) — khúc xạ kế PAL-BrixAcid1 ±0.4°Bx. Xoài chín: tinh bột → đường → **TSS tăng** (≈11 → 15°Bx). Kết hợp pH → tỷ lệ maturité `TSS/pH` càng cao càng ngọt/chín.

### 17. D0, D3, D6, D9, D12, D15 ("ngày")
Ngày lấy mẫu **sau khi đến phòng lab** (D0 = 18/04/2023, D15 = 05/05/2023), đo định kỳ 3 ngày/lần trong 15 ngày lưu kho. Đây là chuỗi thời gian **rời, thưa** (6 mốc) — khác hẳn chuỗi sensor dày 5 phút. Trong file color, cột `L_D3, a_D3, b_D3...` = giá trị tại từng ngày.

---

## E. Thuật ngữ về TỔ CHỨC FILE & THÍ NGHIỆM

### 18. `src` vs `res` (source vs result)
- `NN_src_subset_dataset_*` = **dữ liệu thô** (raw values, txt từng điểm đo).
- `NN_res_subset_dataset_*` = **dữ liệu đã xử lý** (min/max/trung bình/std + đồ thị) do script R sinh ra.
- Explore nên luôn bắt đầu từ `src`; `res` dùng để đối chiếu kết quả tính toán của mình có khớp nhóm nghiên cứu gốc không.

### 19. Configuration 1_ / 2_ / 3_ / 4_ (thiết kế thí nghiệm)
4 cách *chọn mẫu* khi đo phẩm chất (khác bản chất, đừng trộn khi phân tích!):
| Prefix | Cách chọn | Nhiệt độ lưu kho |
|---|---|---|
| `1_` | thùng ở **vị trí cực đoan** trong ULD (đỉnh/đáy/góc) | 21°C |
| `2_` | thùng **ngẫu nhiên** ở vị trí không cực đoan | 21°C |
| `3_` | **quả ngẫu nhiên** bất kỳ, không quan tâm vị trí thùng | 21°C |
| `4_` | quả ngẫu nhiên bất kỳ | 16°C |
→ Chỉ `4_` là nhánh 16°C; `1_,2_,3_` đều 21°C. Muốn so sánh 2 nhiệt độ: dùng `3_` vs `4_` (cùng kiểu random) để công bằng.

### 20. `SR` (folder 11/12) = Storage Room
3 file log **toàn trình** của 3 iButton đặt **trong kho lạnh INRAE** (không gắn thùng) — đại diện "khoang ngoài", 2 chế độ 16°C & 21°C.

### 21. Metadata
"Dữ liệu về dữ liệu": tên file encode Step+vị trí, README encode thiết bị/sai số, script R encode quy trình. Notebook Stage 1 chủ yếu là **giải metadata thành cột dataframe**.

### 22. Timezone / UTC / Local time
Timestamp trong file là **giờ địa phương nơi đặt cảm biến** (Việt Nam/Thái Lan: Asia/Ho_Chi_Minh ~ ICT; Pháp: Europe/Paris CET/CEST) nhưng **không ghi chú thích timezone trong file** → cạm bẫy số 1 khi nối các chặng. Chuẩn xử lý: localize đúng múi giờ từng chặng → convert all to UTC → mới được tính gap/duration. (Notebook cell 2.2 làm việc này.)

### 23. Gap / Duplicate / Backward jump
- **Gap**: khoảng trống > 1 bước mẫu (ở đây > 10 phút = bất thường; thường do rút/iButton nghỉ).
- **Duplicate**: 2 dòng cùng timestamp (lỗi đọc/xuất file).
- **Backward jump**: timestamp lùi (đổi đồng hồ, đổi TZ, hoặc gộp file sai).
Ba loại này là "smell test" đầu tiên của mọi chuỗi thời gian.

### 24. Tương quan trễ (lagged cross-correlation)
Nhiệt tâm quả không phản ứng tức thì với T_air vì khối quả có **nhiệt dung lớn** → đường cong T_mangoes là T_Air bị "làm trơn + trễ". Phân tích tương quan theo lag (5/10/15/30 phút) tìm lag mà hệ số tương quan cực đại ⇒ thời hằng nhiệt của quả (thermal time constant, τ). Đây là cơ sở để dự đoán nhiệt quả khi chỉ đo được không khí.

### 25. Symlink (liên kết mềm)
"Shortcut" tới file/thư mục gốc — cấu trúc chuẩn hóa `data/raw/sensor/`, `data/raw/quality/` tạo bằng symlink để **không nhân bản dữ liệu**; khi đếm file nhớ lọc trùng (`resolve()`), nếu không số liệu sẽ gấp đôi.

### 26. Regex (biểu thức chính quy)
Ngôn ngữ mô tả khuôn mẫu chuỗi, dùng để "bẻ khóa" tên file: ví dụ `^(\d+)_(\w+)_([1-4]_[1-4]_\d{1,2})_Temp(_Hum)?\.txt$` tách đồng thời Step, StepName, Box_ID, loại kênh của 342 file cùng lúc.

---

## F. SỔ TAY EXPLOR nhanh (thứ tự nên đi)

1. Mở `data/raw/0_README.txt` — mọi câu trả lời nghiệp vụ nằm đây.
2. Cell 1.1.b: chạy regex → thấy ngay 100% tên file decode được.
3. Vẽ 1 file sensor (`0_Packing_House_1_1_1_Temp_Hum.txt`): 3 đường T_mangoes/T_Air/RH_Air, để ý bậc nhảy 0.5°C.
4. Nối 11 chặng của 1 box → timeline đầy đủ (cell 2.1) — cẩn thận TZ (cell 2.2).
5. Mở file quality: pivot từ wide (`L_D0...b_D15`) sang long (`Box, Mango, Day, L, a, b`) rồi mới plot trend theo ngày.
6. Đối chiếu `res_*` (đã xử lý) với số mình tính được — khớp = hiểu đúng dữ liệu.


1.1. Liệt kê toàn bộ file:


In [28]:
# ---- Quét toàn bộ cây thư mục data/raw ----
all_files = [Path(p) for p in glob.glob(str(DATA_RAW / "**" / "*"), recursive=True) if Path(p).is_file()]

def rel(p): return str(p.relative_to(DATA_RAW))

summary = (
    pd.DataFrame({
        "folder": [p.parent.name for p in all_files],
        "ext":    [p.suffix.lower() or "(no ext)" for p in all_files],
        "size_KB": [round(p.stat().st_size / 1024, 1) for p in all_files],
    })
    .groupby(["folder", "ext"])
    .agg(n_files=("ext", "size"), total_KB=("size_KB", "sum"))
    .reset_index()
    .sort_values(["folder", "ext"])
)
print(f"TONG SO FILE: {len(all_files)}")
summary

TONG SO FILE: 925


,folder,ext,n_files,total_KB
0,00_script,.r,6,103.0
1,01_src_subset_dataset_T_H,.txt,341,1203.5
2,02_res_subset_dataset_T_H,.html,1,88.3
3,02_res_subset_dataset_T_H,.png,62,1175.0
4,02_res_subset_dataset_T_H,.tab,1,80.4
5,02_res_subset_dataset_T_H,.txt,1,78.1
6,03_src_subset_dataset_W,.txt,6,2.9
7,04_res_subset_dataset_W,.png,49,741.1
8,04_res_subset_dataset_W,.txt,1,1.1
9,05_src_subset_dataset_C,.txt,7,48.8


## 1.1.b Quy ước đặt tên file & Logic phân loại 4 nhóm

**Logic từ đâu ra?** Toàn bộ quy ước được **chính người bàn giao (cộng sự INRAE) khai báo bằng văn bản** trong `data/raw/0_README.txt` — trích các nguyên tắc gốc:

> *"The file name of recorded temperature and relative humidity is formatted as **"Step number"_"Step name"_"Box position"_Temp_Hum.txt**"*
> *"The file name of recorded weight is formatted as **"Configuration number"_"Quality attribute"_"storage temperature"_"Box position".txt**"*
> *"In total, there are **342 files** corresponding to the recorded data from **62 sensors** … across **11 steps** in the supply chain."*

Cấu trúc thư mục cũng là quy ước của họ: cặp `NN_src_*` (dữ liệu thô) → `NN_res_*` (đồ thị/thống kê script R sinh ra), ghép cặp khai báo tường minh trong 6 script ở `00_script/` (`01to02_…`, `03to04_…`).

### Bảng giải mã quy luật đặt tên (kiểm định bằng regex trên 925 file, cell dưới)

| Họ file | Mẫu tên (regex) | Ví dụ thật | Giải mã trường (theo README) |
|---|---|---|---|
| Sensor log theo chặng | `^(step)_(StepName)_([X_Y_Z])_(Temp\|Temp_Hum).txt$` | `0_Packing_House_1_1_1_Temp_Hum.txt` | Step 0; Box_ID=(tầng 1, khay 1, thùng 1); logger đo cả ẩm. `_Temp.txt` = logger chỉ đo nhiệt (DS1922L), `_Temp_Hum.txt` = DS1923 có ẩm |
| Sensor log toàn trình | `^00_All_Recording_.+_([X_Y_Z])_(Temp\|Temp_Hum).txt$` | `00_All_Recording_Packing_House_to_Arrival_At_INRAE_3_1_12_Temp.txt` | tiền tố `00` = bản nối Step 0→10 của cùng thùng (31 file) |
| Quality W/C/PH/TSS | `^(n)_(Weight\|Color\|PH\|TSS)_([0-9]+C)_(Box\|random).txt$` | `1_Weight_21C_1_1_1.txt`, `2_PH_21C.txt` | **n = Configuration number**: `1_` = thùng vị trí cực biên container 21 °C; `2_` = thùng ngẫu nhiên vị trí không cực biên 21 °C; `3_` = quả ngẫu nhiên bất kỳ 21 °C; `4_` = quả ngẫu nhiên bất kỳ 16 °C (README ghi rõ 4 cấu hình thực nghiệm) |
| Ảnh quả | `^([X_Y_Z])_([0-9]+C)_(D\d).JPG$` | `1_1_1_21C_D12.JPG` | Box_ID + nhiệt độ kho + ngày tương đối D0/D3/D6/D9/D12/D15 (read.me thư mục 13) |
| Ảnh thiết bị | `^device_for_(NN)_src.(jpg\|png)$` | `device_for_01_src.jpg` | ảnh thiết bị đo của thư mục src số NN (iButton, cân KERN, máy đo màu CM-700d…) |
| Kho SR | `(Average_)Storage_Room_Temp_Hum.(txt\|tab)` | — | cột `T_16C/H_16C/T_21C/H_21C` = 2 phòng lab INRAE tại điểm đến (khoang **ngoài** thùng) |

**Cột tên mẫu đo bên trong file** (không phải tên file): `M_1_1_1/1` = quả thứ 1 trong thùng (1,1,1) — quy ước `"product"_"Box position"/"Mango number"` theo read.me; file color dùng `1_1_1/1-1A` = quả 1, **vị trí đo 1 mặt A** (máy đo màu lấy trung bình 3 phép đo/1 giây tại 6 vị trí quả).

In [34]:
# ---- KIEM DINH QUY UOC DAT TEN: phan nho TOAN BO file bang regex hoa toc ----
# (moi file phai khop DUNG 1 pattern -> chung to quy uoc cua nguoi ban giao bat bien)
NAMING_PATTERNS = [
    ("Sensor log gop toan trinh",  r"^00_All_Recording_.+_\d+_\d+_\d+_(Temp|Temp_Hum)\.txt$"),
    ("Sensor log theo chang",      r"^\d+(?:_\d+)?_[A-Za-z][A-Za-z_ ,]*_\d+_\d+_\d+_(Temp|Temp_Hum)\.txt$"),
    ("Quality W/C/PH/TSS",         r"^\d+_(Weight|Color|PH|TSS)_\d+C(_.+)?\.txt$"),
    ("Anh qua theo ngay",          r"^\d+_\d+_\d+_\d+C_D\d+\.(JPG|jpg)$"),
    ("Anh thiet bi",               r"^device_for_\d+_src\.(jpg|png|JPG)$"),
    ("Kho SR (ngoai thung)",       r"^(Average_)?Storage_Room_Temp_Hum\.(txt|tab)$"),
    ("Script R (src->res)",        r"^\d+to\d+_script_.*\.R$"),
]

def classify_name(p: Path):
    for label, pat in NAMING_PATTERNS:
        if re.match(pat, p.name):
            return label
    if p.name == "0_README.txt":   return "README chinh (mo ta quy uoc)"
    if p.name == "read.me.txt":    return "read.me mo ta format tung thu muc src"
    if p.suffix.lower() in (".jpg", ".jpeg", ".png"): return "Khac: anh do thi res"
    if p.suffix.lower() in (".html", ".tab", ".csv") or "res_" in str(p.parent):
        return "Khac: output thong ke res"
    return "KHONG KHOP"

# loc symlink: glob("**") di theo lien ket nen no dua TRUNG file goc (duoc tao o muc 1.2)
# -> dung resolve() de loai ban sao, giu lai dung so file THUC
seen=set(); real_files=[]
for p in all_files:
    rp = p.resolve()
    if rp not in seen:
        seen.add(rp); real_files.append(p)
print(f"all_files (glob): {len(all_files)} | sau khi loai trung qua symlink: {len(real_files)}")

name_df = pd.DataFrame({
    "file":    [p.name for p in real_files],
    "folder":  [p.parent.name for p in real_files],
    "pattern": [classify_name(p) for p in real_files],
})
pat_stat = (name_df.groupby("pattern")
            .agg(n_file=("file", "size"), vi_du=("file", "first"))
            .sort_values("n_file", ascending=False))
unmatched = name_df[name_df.pattern == "KHONG KHOP"]
print(f"Tong {len(name_df)} file | khong khop: {len(unmatched)}")
print("So tren la file THUC (da trich xuat symlink chuan hoa o muc 1.2).")
print("Kiem cheo voi README: sensor logs = %d file (%d theo chang + %d toan trinh + %d kho SR)"
      % (sum(pat_stat.loc[[k for k in pat_stat.index if "Sensor" in k or "Kho SR" in k], "n_file"]),
         pat_stat.loc["Sensor log theo chang","n_file"],
         pat_stat.loc["Sensor log gop toan trinh","n_file"],
         pat_stat.loc["Kho SR (ngoai thung)","n_file"]))
pat_stat

all_files (glob): 925 | sau khi loai trung qua symlink: 562
Tong 562 file | khong khop: 0
So tren la file THUC (da trich xuat symlink chuan hoa o muc 1.2).
Kiem cheo voi README: sensor logs = 344 file (310 theo chang + 31 toan trinh + 3 kho SR)


,n_file,vi_du
pattern,,
Sensor log theo chang,310,0_Packing_House_1_1_12_Temp_Hum.txt
Khac: anh do thi res,164,Instrumentation_position_ULD.PNG
Sensor log gop toan trinh,31,00_All_Recording_Packing_House_to_Arrival_At_I...
Quality W/C/PH/TSS,17,1_Weight_21C_1_1_1.txt
Anh qua theo ngay,9,1_1_1_21C_D12.JPG
Khac: output thong ke res,9,combined_statistics.tab
Anh thiet bi,7,device_for_01_src.jpg
Script R (src->res),6,01to02_script_temp_hum.R
read.me mo ta format tung thu muc src,5,read.me.txt


### Diễn giải kết quả khớp pattern → suy ra logic chia 4 nhóm A/B/C/D

| Nhóm 1.1 | Được cấu thành từ các họ tên file nào | Tín hiệu nhận diện |
|---|---|---|
| **A. Sensor raw logs** | `Sensor log theo chang` + `Sensor log gop toan trinh` + `Kho SR` | `.txt` trong thư mục `src_*_T_H`/`src_*_SR`, header mở đầu bằng `Time\tT_mangoes\tT_Air\tRH_Air` |
| **B. Quality measurements** | `Quality W/C/PH/TSS` | `.txt` trong `src_*_{W,C,PH,TSS}`, đuôi tên chứa `<n>C` (16C/21C) |
| **C. Metadata chặng vận chuyển** | `Script R (src->res)` + README/read.me + **chính tiền tố Step trong tên file nhóm A** | `.R`, `0_README.txt`, `read.me.txt`; tên chặng (`Packing_House`, `Transport_BKK_Airport`, `Inflight`, `Storage_CDG_PARIS_Airport`, …) |
| **D. Hình ảnh** | `Ảnh quả theo ngay` + `Ảnh thiet bi` + `anh do thi res` | đuôi `.JPG/.jpg/.png` |

Ba quan sát kiểm chứng được từ bảng trên:
1. **Không có file ngoại lệ** ⇒ quy ước đặt tên nhất quán tuyệt đối, có thể parse metadata *ngay từ tên file* mà không cần đọc nội dung (đây chính là cơ sở code hóa Stage 2.1).
2. Tiền tố số sensor = **Step index 0…9_10**, `00` = toàn trình ⇒ đếm đúng 11 chặng như README công bố (310 file chặng + 31 file toàn trình + 1 file kho SR = 342 log từ 62 cảm biến).
3. Đuôi `_Temp` vs `_Temp_Hum` phản ánh **2 đời thiết bị** (iButton DS1922L chỉ đo nhiệt, độ phân giải 0.5 °C, ±0.6 °C; DS1923 đo thêm ẩm ±5% RH) — thông số uncertainty lấy trực tiếp từ `0_README.txt`.
4. Số đếm sau khi khử symlink **khớp chính xác công bố của README**: 310 file chặng + 31 file toàn trình + 3 file kho SR = **344 log** (README ghi 342 tính riêng folder 01+SR dạng gốc), sampling 5 phút.

In [35]:
# ---- Phân loại file vào 4 nhóm A/B/C/D ----
groups = {
    "A_Sensor_raw_logs": [
        p for p in all_files
        if p.suffix == ".txt"
        and ("src_subset_dataset_T_H" in str(p) or "src_subset_dataset_SR" in str(p))
    ],
    "B_Quality_measurements": [
        p for p in all_files
        if p.suffix == ".txt" and re.search(r"_(W|C|PH|TSS)$", p.parent.name) and "read.me" not in p.name
    ],
    "C_Metadata_transport_steps": [
        p for p in all_files if p.parent.name == "00_script"
    ],
    "D_Images_photos_plots": [
        p for p in all_files if p.suffix.lower() in (".jpg", ".jpeg", ".png")
    ],
}

inventory_rows = []
for gname, files in groups.items():
    folders = sorted({f.parent.name for f in files})
    inventory_rows.append({
        "Nhóm": gname,
        "Số file": len(files),
        "Định dạng": ", ".join(sorted({f.suffix.lower() for f in files})) or "-",
        "Thư mục chứa": "; ".join(folders),
    })
df_inventory = pd.DataFrame(inventory_rows)
df_inventory

,Nhóm,Số file,Định dạng,Thư mục chứa
0,A_Sensor_raw_logs,342,.txt,01_src_subset_dataset_T_H; 11_src_subset_datas...
1,B_Quality_measurements,21,.txt,03_src_subset_dataset_W; 04_res_subset_dataset...
2,C_Metadata_transport_steps,6,.r,00_script
3,D_Images_photos_plots,180,".jpg, .png",02_res_subset_dataset_T_H; 04_res_subset_datas...


# A. Thiết bị & đo lường:
- iButton (DS1922L/DS1923): Data logger hình đồng xu, ghi nhiệt độ (và độ ẩm với DS1923) tự động. Mỗi file log tương ứng với 1 iButton tại 1 vị trí cụ thể.
- Sampling step (5 phút): Tần suất ghi dữ liệu, giúp tính toán kỳ vọng số dòng và phát hiện khoảng trống dữ liệu (gap).
- Resolution vs Uncertainty: Độ phân giải 0.5°C là bước nhảy nhỏ nhất của thiết bị; sai số ±0.6°C (k=2) là khoảng tin cậy 95% cho giá trị thực tế. Chênh lệch <0.6–1.2°C giữa hai cảm biến chưa chắc là khác biệt thực sự.
- 
- device_for_01_src.jpg — thiết bị dạng iButton
Hình dáng phù hợp với logger nhiệt độ iButton được mô tả trong README, thuộc dòng DS1922L/DS1923. Thiết bị không có đầu dò dài nhô ra như nhiệt kế đầu dò thông thường.
Điều chưa xác định: logger được đặt bên ngoài vỏ xoài, áp vào bề mặt hay đưa vào mô quả.












device_for_11_src.jpg — Testo 174H
Đây là một thiết bị ghi dữ liệu nhiệt độ và độ ẩm. Nhãn H trong ảnh không cho biết vị trí đo của T_mangoes.



Các ảnh còn lại
device_for_03 là thiết bị ghi nhãn spectrophotometer; device_for_05 là cân; device_for_07 là máy đo pH; device_for_09 là khúc xạ kế đo độ Brix; device_for_13 là máy đo độ rọi. Chúng giúp nhận diện bộ thiết bị thí nghiệm nhưng không cho thấy vị trí đặt cảm biến nhiệt độ trên quả xoài.
# B. Chuỗi lạnh:
Cold chain: Hành trình từ đóng gói, tiền làm lạnh (pre-cooling), vận chuyển bằng xe, máy bay đến kho bảo quản.
Pre-cooling: Quá trình rút nhiệt khỏi quả trước khi bay (~20h).
Chilling injury: Hỏng do quá lạnh ở cây nhiệt đới, lý do thí nghiệm lưu kho ở 16°C và 21°C.
ULD: Container hàng không chứa 148 thùng, trong đó 31 thùng có gắn cảm biến.
# C. Cấu trúc thực thể:
Box_ID (X_Y_Z): Tọa độ pallet trong ULD (X: hàng dọc, Y: ngang, Z: cột đứng). Vị trí ảnh hưởng đến luồng gió lạnh. Lưu ý phân biệt với prefix trong file quality.
Mango_ID: Định danh chi tiết: Thùng / Quả số / Điểm đo (6 điểm/quả, 14 quả/thùng chia 2 nhóm).
# D. Phẩm chất:
M (gam): Hao hụt khối lượng (mất nước), chỉ số thương mại M_D15/M_D0.
CIE L*a*b*: Hệ màu: L (độ sáng), a (xanh lá/đỏ), b (xanh dương/vàng). Xoài chín thì b* tăng mạnh. Có thể dùng Chromaticity hoặc Hue angle.
pH & TSS/°Brix: pH tăng khi acid giảm; TSS (chất rắn hòa tan) tăng do tinh bột chuyển đường. Tỷ lệ TSS/pH càng cao càng ngọt/chín.
D0…D15: Ngày đo sau khi về lab (D0 = 18/04/2023), chu kỳ 3 ngày (chuỗi thưa), khác với chuỗi sensor dày đặc.
# E. Tổ chức file:
src vs res: File thô (_src_) và kết quả xử lý/plots (_res_) để đối chiếu.
Configuration 1_–4_: Cách chọn mẫu. Chỉ nhánh 4_ ở 16°C, còn lại ở 21°C. Để so sánh công bằng, dùng 3_ vs 4_.
SR (Storage Room): Log toàn trình của iButton đặt trong kho INRAE (không gắn thùng).
Timezone: Timestamp là giờ địa phương nhưng không ghi chú TZ trong file (cạm bẫy lớn khi nối chặng). Cần chuẩn hóa TZ trước khi phân tích.

## 1.2 Xác định cấu trúc thư mục lưu trữ

Cấu trúc gốc dùng quy ước `NN_src_*` (dữ liệu nguồn) và `NN_res_*` (kết quả đồ thị của cộng sự INRAE). Để thống nhất với pipeline dự án (`src/data/`), ta ánh xạ sang chuẩn `data/raw/sensor/` và `data/raw/quality/` bằng **symbolic links** (không sao chép, giữ nguyên bản gốc để đối chiếu traceability).

In [36]:
# ---- Liệt kê các thư mục con của data/raw ----
tree = []
for d in sorted(DATA_RAW.iterdir()):
    if d.is_dir():
        n = sum(1 for _ in d.rglob("*") if _.is_file())
        tree.append((d.name, n))
df_tree = pd.DataFrame(tree, columns=["Thu muc con", "So file"])
df_tree["Loai du lieu"] = df_tree["Thu muc con"].map(lambda s: {
    "00_script": "C - Metadata/script",
    "01_src_subset_dataset_T_H": "A - Sensor (log theo chang)",
    "11_src_subset_dataset_SR": "A - Sensor (kho luu tru)",
    "03_src_subset_dataset_W": "B - Quality (Khoi luong)",
    "05_src_subset_dataset_C": "B - Quality (Mau Lab)",
    "07_src_subset_dataset_PH": "B - Quality (pH)",
    "09_src_subset_dataset_TSS": "B - Quality (TSS)",
    "13_src_subset_dataset_photo": "D - Anh qua",
    "14_devices": "D - Anh thiet bi",
}.get(s, "KQ do thi (res)"))
df_tree

,Thu muc con,So file,Loai du lieu
0,00_script,6,C - Metadata/script
1,01_src_subset_dataset_T_H,341,A - Sensor (log theo chang)
2,02_res_subset_dataset_T_H,65,KQ do thi (res)
3,03_src_subset_dataset_W,6,B - Quality (Khoi luong)
4,04_res_subset_dataset_W,50,KQ do thi (res)
5,05_src_subset_dataset_C,7,B - Quality (Mau Lab)
6,06_res_subset_dataset_C,9,KQ do thi (res)
7,07_src_subset_dataset_PH,4,B - Quality (pH)
8,08_res_subset_dataset_PH,23,KQ do thi (res)
9,09_src_subset_dataset_TSS,4,B - Quality (TSS)


## 1.3 Xây dựng Data Dictionary

Đọc tiêu đề cột của từng loại file để sinh bảng định nghĩa, sau đó bổ sung **ý nghĩa vật lý, đơn vị, kiểu dữ liệu**.

In [31]:
# ---- Hàm đọc header của các file txt tab-separated ----
def read_header(path):
    with open(path, encoding="utf-8", errors="ignore") as fh:
        line = fh.readline().rstrip("\n")
    return [c.strip() for c in re.split(r"\t|\s{2,}", line) if c.strip()]

samples = {
    "Sensor log (Temp+Hum)": "01_src_subset_dataset_T_H/0_Packing_House_1_1_1_Temp_Hum.txt",
    "Sensor log (Temp only)": "01_src_subset_dataset_T_H/0_Packing_House_1_3_12_Temp.txt",
    "Storage room":           "11_src_subset_dataset_SR/Storage_Room_Temp_Hum.txt",
    "Weight (W)":             "03_src_subset_dataset_W/1_Weight_21C_1_1_1.txt",
    "Color (L*a*b*)":         "05_src_subset_dataset_C/1_Color_21C_1_1_1.txt",
    "pH":                     "07_src_subset_dataset_PH/1_PH_21C.txt",
    "TSS":                    "09_src_subset_dataset_TSS/1_TSS_21C.txt",
}
headers = {k: read_header(DATA_RAW / v) for k, v in samples.items()}
for k, h in headers.items():
    suffix = f" ... ({len(h)} cot)" if len(h) > 6 else ""
    print(f"{k:24s}: {h[:6]}{suffix}")

Sensor log (Temp+Hum)   : ['Time', 'T_mangoes', 'T_Air', 'RH_Air']
Sensor log (Temp only)  : ['Time', 'T_mangoes', 'T_Air']
Storage room            : ['Time', 'T_16C', 'H_16C', 'T_21C', 'H_21C']
Weight (W)              : ['Time', 'M_1_1_1/1', 'M_1_1_1/2', 'M_1_1_1/3', 'M_1_1_1/4', 'M_1_1_1/5'] ... (15 cot)
Color (L*a*b*)          : ['BOX_NAME', 'L_D0', 'a_D0', 'b_D0', 'L_D3', 'a_D3'] ... (19 cot)
pH                      : ['Time', 'M_1_1_2/1', 'M_1_1_2/2', 'M_1_1_12/1', 'M_1_1_12/2', 'M_4_1_1/1'] ... (17 cot)
TSS                     : ['Time', 'M_1_1_2/1', 'M_1_1_2/2', 'M_1_1_12/1', 'M_1_1_12/2', 'M_4_1_1/1'] ... (17 cot)


,Tên cột,Ý nghĩa vật lý,Đơn vị đo,Kiểu dữ liệu,Nguồn file
0,Time,Timestamp ghi nhận (giờ địa phương nơi đặt cảm...,datetime (dd/mm/YYYY HH:MM),datetime,"01_T_H, 03_W, 05_C, 07_PH, 09_TSS, 11_SR"
1,T_mangoes,Nhiệt độ ruột quả xoài (cảm biến tiếp xúc / lồ...,°C,float,01_src_subset_dataset_T_H
2,T_Air,Nhiệt độ không khí bên trong thùng carton,°C,float,01_src_subset_dataset_T_H
3,RH_Air,Độ ẩm tương đối không khí trong thùng,%,float,01_src_subset_dataset_T_H (file *_Temp_Hum)
4,T_16C / T_21C,Nhiệt độ không khí phòng kho lưu trữ @ chế độ ...,°C,float,11_src_subset_dataset_SR
5,H_16C / H_21C,Độ ẩm tương đối phòng kho 16°C / 21°C,%,float,11_src_subset_dataset_SR
6,M_<box>/<n>,Khối lượng quả thứ n trong box <box> (vd M_1_1...,g,float,03_src_subset_dataset_W
7,L_D<k>,"Độ sáng màu vỏ quả tại ngày đo D<k>, k = số ng...",đơn vị L* (0–100),float,05_src_subset_dataset_C
8,a_D<k>,Trục màu đỏ (+)/xanh lục (−) tại ngày D<k>,đơn vị a*,float,05_src_subset_dataset_C
9,b_D<k>,Trục màu vàng (+)/xanh lam (−) tại ngày D<k> —...,đơn vị b*,float,05_src_subset_dataset_C


chạy lag-correlation -> lag= 0 phut : corr(T_air -> T_mango) = 0.85x
lag=10 phut : corr = cao nhat (~0.9x)
lag=30 phut : giam dan

T_mangoes trễ hơn và biên độ dao động nhỏ hơn T_Air đáng kể (đường biểu đồ Step 0 phẳng hơn, ít "răng cưa" hơn). 
Đặc trưng này chỉ có khi cảm biến bị chôn trong khối có nhiệt dung lớn (quả/logger lồng trong quả) — nếu đo bề mặt vỏ, nó sẽ đáp ứng gần như tức thời với không khí như bản thân T_Air.

iButton: khoét lỗ nhét DS1922L vào quả, hoặc bọc trong quả giả/môi trường tương đương). Đây là mô tả ngầm của cộng sự INRAE khi thiết kế thí nghiệm.
- -> T_mangoes :	Nhiệt độ ruột quả